# ResuMatch AI: Multi-Agent HR Pipeline (Google ADK)
This notebook implements an automated, multi-agent HR screening pipeline using the **Google Agent Development Kit (ADK)** and Google Gemini models.

### How it works:
1. **Extraction Agent**: A specialized LLM agent parses raw resume text (from PDF or Text) and structures it into a clean JSON schema.
2. **Evaluation Agent**: Another specialized agent aligns the extracted candidate profile against a Job Description (JD), calculates a match score, creates a match/gap alignment matrix, and designs tailored interview questions.
3. **Interactive Dashboard**: IPython widgets and HTML render a beautiful comparison of candidate profiles and their evaluation results, and outputs a standalone Markdown report.

In [ ]:
# Install required libraries
%pip install -q google-adk pypdf pandas ipywidgets

import os
import json
import asyncio
from pathlib import Path
import pandas as pd
import pypdf
from IPython.display import display, HTML, Markdown
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types

# Configure Gemini API Key
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        os.environ["GEMINI_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY")
        print("Successfully authenticated using Kaggle Secrets.")
    except Exception as e:
        print(f"Warning: Kaggle Secrets could not be loaded: {e}")
        print("Please ensure your GEMINI_API_KEY is defined in Add-ons -> Secrets.")
else:
    # Try loading from local secrets folder first
    secrets_file = Path("secrets/gemini_key.txt")
    if secrets_file.exists():
        with open(secrets_file, "r") as f:
            key = f.read().strip()
            if key and "YOUR_GEMINI_API_KEY_HERE" not in key:
                os.environ["GEMINI_API_KEY"] = key
                print("API key loaded from secrets/gemini_key.txt")
    if "GEMINI_API_KEY" not in os.environ:
        from getpass import getpass
        os.environ["GEMINI_API_KEY"] = getpass("Enter your GEMINI_API_KEY: ")
    print("Gemini API key is configured.")

In [ ]:
def extract_text_from_file(file_path: str) -> str:
    """Extract raw text from a PDF or TXT file."""
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
        
    if path.suffix.lower() == '.pdf':
        try:
            reader = pypdf.PdfReader(path)
            text = ""
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
            return text.strip()
        except Exception as e:
            raise RuntimeError(f"Error reading PDF {file_path}: {e}")
    elif path.suffix.lower() in ['.txt', '.md']:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read().strip()
    else:
        raise ValueError(f"Unsupported file format: {path.suffix}")

print("Text extractor utility defined successfully.")

In [ ]:
# Define Google ADK Agents
# Note: google-adk automatically uses os.environ["GEMINI_API_KEY"] to configure the client.

# Agent 1: Structured Resume Information Extractor
extractor_agent = LlmAgent(
    name="ResumeExtractor",
    model="gemini-2.5-flash",
    instruction=(
        "You are an expert HR Resume Parser. Your job is to extract candidate details from raw resume text "
        "and return a single structured JSON object. "
        "Do not include any markdown styling, conversational filler, or triple backticks in your output. "
        "Output ONLY valid JSON matching this schema:\n"
        "{\n"
        "  \"name\": \"Full Name\",\n"
        "  \"email\": \"Email\",\n"
        "  \"phone\": \"Phone number\",\n"
        "  \"socials\": {\"linkedin\": \"LinkedIn URL/handle\", \"github\": \"GitHub URL/handle\"},\n"
        "  \"experience_years\": 5.5,\n"
        "  \"top_roles\": [\"List of key job titles\"],\n"
        "  \"skills\": {\n"
        "    \"languages\": [\"Python\", \"Java\"],\n"
        "    \"frameworks\": [\"FastAPI\", \"Django\"],\n"
        "    \"databases\": [\"PostgreSQL\", \"Redis\"],\n"
        "    \"cloud_devops\": [\"Docker\", \"AWS\"]\n"
        "  }\n"
        "}"
    )
)

# Agent 2: Candidate Match Evaluator
evaluator_agent = LlmAgent(
    name="CandidateEvaluator",
    model="gemini-2.5-flash",
    instruction=(
        "You are an expert Technical Recruiter. Your task is to evaluate a candidate's structured resume JSON "
        "against a Job Description (JD). Compute a match score (0-100), compare skill alignment, and generate "
        "exactly 3 tailored technical interview questions based on their profile and gaps.\n"
        "Do not include any conversational text or markdown code blocks (like ```json). Return ONLY valid JSON matching this schema:\n"
        "{\n"
        "  \"candidate_name\": \"Candidate Name\",\n"
        "  \"match_score\": 85,\n"
        "  \"alignment_matrix\": [\n"
        "    {\"requirement\": \"Skill/Requirement Name\", \"status\": \"Matched | Partial | Missing\", \"notes\": \"Detail why\"}\n"
        "  ],\n"
        "  \"technical_questions\": [\n"
        "    \"Tailored question 1\",\n"
        "    \"Tailored question 2\",\n"
        "    \"Tailored question 3\"\n"
        "  ]\n"
        "}"
    )
)

print("ADK Agents defined successfully.")

In [ ]:
# Helper to execute ADK Agent in memory and retrieve response text
async def run_agent_prompt(agent, prompt: str) -> str:
    runner = InMemoryRunner(agent=agent, app_name="ResuMatchApp")
    session = await runner.session_service.create_session(
        app_name="ResuMatchApp",
        user_id="hr_user"
    )
    
    user_message = types.Content(
        role="user",
        parts=[types.Part.from_text(text=prompt)]
    )
    
    response_text = ""
    async for event in runner.run_async(
        user_id="hr_user",
        session_id=session.id,
        new_message=user_message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, 'text') and part.text:
                    response_text += part.text
                elif isinstance(part, str):
                    response_text += part
                    
    await runner.close()
    return response_text.strip()

# Run workflow orchestrator function
async def process_candidate(resume_path: str, jd_text: str):
    print(f"Processing candidate resume: {resume_path}...")
    
    # 1. Extract raw text
    raw_text = extract_text_from_file(resume_path)
    
    # 2. Extract structured profile
    extraction_output = await run_agent_prompt(extractor_agent, raw_text)
    
    # Clean output from potential triple backticks (just in case LLM wraps it)
    clean_extraction = extraction_output.strip()
    if clean_extraction.startswith("```json"):
        clean_extraction = clean_extraction[7:]
    elif clean_extraction.startswith("```"):
        clean_extraction = clean_extraction[3:]
    if clean_extraction.endswith("```"):
        clean_extraction = clean_extraction[:-3]
    clean_extraction = clean_extraction.strip()
    
    try:
        profile_json = json.loads(clean_extraction)
    except Exception as e:
        print(f"Error parsing JSON from Extraction Agent: {e}")
        print(f"Raw output: {extraction_output}")
        # Fallback profile
        profile_json = {"name": Path(resume_path).stem.replace('_', ' ').title(), "skills": {}, "top_roles": []}
    
    # 3. Evaluate alignment
    evaluator_prompt = f"Candidate Profile JSON:\n{json.dumps(profile_json, indent=2)}\n\nJob Description:\n{jd_text}"
    evaluation_output = await run_agent_prompt(evaluator_agent, evaluator_prompt)
    
    clean_evaluation = evaluation_output.strip()
    if clean_evaluation.startswith("```json"):
        clean_evaluation = clean_evaluation[7:]
    elif clean_evaluation.startswith("```"):
        clean_evaluation = clean_evaluation[3:]
    if clean_evaluation.endswith("```"):
        clean_evaluation = clean_evaluation[:-3]
    clean_evaluation = clean_evaluation.strip()
    
    try:
        evaluation_json = json.loads(clean_evaluation)
    except Exception as e:
        print(f"Error parsing JSON from Evaluation Agent: {e}")
        print(f"Raw output: {evaluation_output}")
        evaluation_json = {
            "candidate_name": profile_json.get("name", "Unknown"),
            "match_score": 0,
            "alignment_matrix": [],
            "technical_questions": ["Could not generate evaluation report."]
        }
        
    return {
        "profile": profile_json,
        "evaluation": evaluation_json
    }

print("Workflow runner functions defined successfully.")

In [ ]:
# Visualization Dashboard using Premium HTML
def render_dashboard(results: list):
    html_content = """
    <style>
        .resumatch-dashboard {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: #0f172a;
            color: #f8fafc;
            padding: 24px;
            border-radius: 16px;
            max-width: 1200px;
            margin: 20px auto;
        }
        .header {
            text-align: center;
            margin-bottom: 30px;
            border-bottom: 2px solid #334155;
            padding-bottom: 15px;
        }
        .header h1 {
            color: #38bdf8;
            font-size: 2.5em;
            margin: 0;
        }
        .header p {
            color: #94a3b8;
            font-size: 1.1em;
            margin-top: 5px;
        }
        .candidate-grid {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(500px, 1fr));
            gap: 24px;
        }
        .candidate-card {
            background: #1e293b;
            border-radius: 12px;
            border: 1px solid #334155;
            padding: 20px;
            box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.3);
            transition: transform 0.2s;
        }
        .candidate-card:hover {
            transform: translateY(-4px);
        }
        .card-header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            border-bottom: 1px solid #475569;
            padding-bottom: 10px;
            margin-bottom: 15px;
        }
        .candidate-name {
            font-size: 1.4em;
            font-weight: bold;
            color: #f1f5f9;
        }
        .match-badge {
            font-size: 1.2em;
            font-weight: bold;
            padding: 6px 12px;
            border-radius: 9999px;
            color: white;
        }
        .score-high { background-color: #10b981; }
        .score-mid { background-color: #f59e0b; }
        .score-low { background-color: #ef4444; }
        
        .section-title {
            color: #38bdf8;
            font-size: 1.1em;
            text-transform: uppercase;
            letter-spacing: 0.05em;
            margin-top: 15px;
            margin-bottom: 8px;
        }
        .contact-info {
            font-size: 0.9em;
            color: #94a3b8;
            margin-bottom: 15px;
        }
        .skills-container {
            display: flex;
            flex-wrap: wrap;
            gap: 6px;
            margin-bottom: 15px;
        }
        .skill-badge {
            background: #334155;
            color: #e2e8f0;
            padding: 3px 8px;
            border-radius: 4px;
            font-size: 0.85em;
        }
        .matrix-table {
            width: 100%;
            border-collapse: collapse;
            margin-bottom: 15px;
            font-size: 0.9em;
        }
        .matrix-table th {
            text-align: left;
            border-bottom: 2px solid #475569;
            color: #94a3b8;
            padding: 6px;
        }
        .matrix-table td {
            border-bottom: 1px solid #334155;
            padding: 8px 6px;
        }
        .status-badge {
            display: inline-block;
            padding: 2px 6px;
            border-radius: 4px;
            font-size: 0.8em;
            font-weight: bold;
        }
        .status-Matched { background: #064e3b; color: #6ee7b7; }
        .status-Partial { background: #78350f; color: #fde047; }
        .status-Missing { background: #7f1d1d; color: #fca5a5; }
        
        .questions-list {
            margin: 0;
            padding-left: 20px;
            color: #e2e8f0;
            font-size: 0.95em;
        }
        .questions-list li {
            margin-bottom: 6px;
        }
    </style>
    <div class="resumatch-dashboard">
        <div class="header">
            <h1>ResuMatch AI HR Evaluation Dashboard</h1>
            <p>Automated Multi-Agent Recruitment Pipeline</p>
        </div>
        <div class="candidate-grid">
    """
    
    for c in results:
        prof = c["profile"]
        ev = c["evaluation"]
        
        score = ev.get("match_score", 0)
        badge_class = "score-low"
        if score >= 75:
            badge_class = "score-high"
        elif score >= 50:
            badge_class = "score-mid"
            
        contact_parts = []
        if prof.get("email"): contact_parts.append(prof["email"])
        if prof.get("phone"): contact_parts.append(prof["phone"])
        contact_str = " | ".join(contact_parts)
        
        # Collect skills
        skills_badges = []
        skills_dict = prof.get("skills", {})
        if isinstance(skills_dict, dict):
            for cat, items in skills_dict.items():
                if isinstance(items, list):
                    for item in items:
                        skills_badges.append(f'<span class="skill-badge">{item}</span>')
        skills_html = "".join(skills_badges) if skills_badges else '<span class="contact-info">None found</span>'
        
        # Build alignment matrix rows
        matrix_rows = ""
        matrix = ev.get("alignment_matrix", [])
        if isinstance(matrix, list):
            for row in matrix:
                req = row.get("requirement", "")
                stat = row.get("status", "Missing")
                notes = row.get("notes", "")
                matrix_rows += f"""
                <tr>
                    <td>{req}</td>
                    <td><span class="status-badge status-{stat}">{stat}</span></td>
                    <td style="color: #94a3b8;">{notes}</td>
                    </tr>
                """
        else:
            matrix_rows = "<tr><td colspan=\'3\'>No alignment details available</td></tr>"
            
        # Build questions
        questions_html = ""
        questions = ev.get("technical_questions", [])
        if isinstance(questions, list):
            for q in questions:
                questions_html += f"<li>{q}</li>"
        else:
            questions_html = f"<li>{questions}</li>"
            
        html_content += f"""
        <div class="candidate-card">
            <div class="card-header">
                <div class="candidate-name">{prof.get("name", "Unknown Candidate")}</div>
                <div class="match-badge {badge_class}">{score}% Match</div>
            </div>
            <div class="contact-info">{contact_str}</div>
            
            <div class="section-title">Key Profile Info</div>
            <div style="font-size: 0.9em; color: #cbd5e1; margin-bottom: 8px;">
                Experience: <strong>{prof.get("experience_years", "N/A")} years</strong><br/>
                Top Roles: {', '.join(prof.get("top_roles", []))}
            </div>
            
            <div class="section-title">Extracted Skills</div>
            <div class="skills-container">
                {skills_html}
            </div>
            
            <div class="section-title">JD Alignment Matrix</div>
            <table class="matrix-table">
                <thead>
                    <tr>
                        <th>Requirement</th>
                        <th>Status</th>
                        <th>Notes</th>
                    </tr>
                </thead>
                <tbody>
                    {matrix_rows}
                </tbody>
            </table>
            
            <div class="section-title">Tailored Interview Questions</div>
            <ul class="questions-list">
                {questions_html}
            </ul>
        </div>
        """
        
    html_content += """
        </div>
    </div>
    """
    display(HTML(html_content))
    
    # Save a copy as Markdown
    md_content = "# ResuMatch AI HR Evaluation Report\n\n"
    for c in results:
        prof = c["profile"]
        ev = c["evaluation"]
        md_content += f"## Candidate: {prof.get('name', 'Unknown')} - Match Score: {ev.get('match_score', 0)}%\n"
        md_content += f"- **Email:** {prof.get('email', 'N/A')}\n"
        md_content += f"- **Experience:** {prof.get('experience_years', 'N/A')} years\n"
        md_content += f"- **Top Roles:** {', '.join(prof.get('top_roles', []))}\n\n"
        md_content += "### Skill Alignment Matrix\n\n"
        md_content += "| Requirement | Status | Notes |\n"
        md_content += "| --- | --- | --- |\n"
        for row in ev.get("alignment_matrix", []):
            md_content += f"| {row.get('requirement')} | {row.get('status')} | {row.get('notes')} |\n"
        md_content += "\n### Tailored Interview Questions\n"
        for q in ev.get("technical_questions", []):
            md_content += f"- {q}\n"
        md_content += "\n---\n\n"
        
    with open("data/evaluation_report.md", "w", encoding="utf-8") as f:
        f.write(md_content)
    print("Report saved to data/evaluation_report.md")

print("Dashboard visualization functions defined successfully.")

In [ ]:
# Execution Cell
async def run_pipeline():
    jd_text = extract_text_from_file("data/job_descriptions/backend_engineer.txt")
    resumes_dir = Path("data/resumes")
    
    results = []
    for file in resumes_dir.iterdir():
        if file.suffix.lower() in ['.txt', '.pdf']:
            try:
                res = await process_candidate(str(file), jd_text)
                results.append(res)
            except Exception as e:
                print(f"Error processing {file.name}: {e}")
            
    render_dashboard(results)

# Execute the pipeline
await run_pipeline()